# Fashion Outfit Recommender — Colab

Chạy theo thứ tự: mount Drive → tải Polyvore → tiền xử lý Polyvore → huấn luyện embedding, luật kết hợp, compatibility và ranking. Chọn **Runtime → Change runtime type → T4 GPU** trước khi chạy.

Dữ liệu và checkpoint được lưu trên Drive để có thể tiếp tục. Notebook tự tìm thư mục dự án trong My Drive hoặc shortcut của thư mục được chia sẻ. Dataset Polyvore trên Hugging Face có thể yêu cầu quyền truy cập được chấp thuận và đăng nhập.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_CANDIDATES = [
    Path('/content/drive/.shortcut-targets-by-id/1f2WLaxnP-_c_7pJF0kRJYkAFPCrGR9VR/fashion-outfit-recommender'),
    Path('/content/drive/MyDrive/fashion-outfit-recommender'),
]
PROJECT = next((p for p in PROJECT_CANDIDATES if (p / 'scripts/download_datasets.py').is_file()), None)
assert PROJECT is not None, 'Không tìm thấy project. Kiểm tra shortcut fashion-outfit-recommender trong Drive.'
print('Project:', PROJECT)
%cd $PROJECT
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Hãy bật GPU trong Runtime > Change runtime type rồi chạy lại.')

## 1. Cài thư viện
Giữ PyTorch/torchvision bản CUDA có sẵn trong Colab; chỉ cài các thư viện còn lại.

In [ ]:
%pip install -q pandas numpy scikit-learn pyarrow mlxtend huggingface_hub pillow requests joblib pyyaml
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'

## 2. Đăng nhập Hugging Face và tải/kiểm tra dataset
Nếu Polyvore đã tải đủ theo manifest, script sẽ xác minh và không tải lại. Không dán token vào notebook đã chia sẻ.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # cần quyền truy cập dataset Polyvore
!python -u scripts/download_datasets.py --dataset polyvore

## 3. Tiền xử lý Polyvore
Giữ nguyên dữ liệu gốc; đầu ra nằm trong `data/processed/polyvore/` cùng báo cáo số lượng.

In [ ]:
!python -u scripts/preprocess_polyvore.py

## 4. Huấn luyện đầy đủ và đánh giá
Chạy lần lượt: FP-Growth so sánh Apriori, Logistic Regression so sánh Random Forest trên metadata, đánh giá baseline gợi ý, ResNet-50 + Triplet Loss, xuất embedding, compatibility có ảnh và ranking bằng MMR. Thời gian phụ thuộc GPU, dữ liệu và giới hạn Colab. Checkpoint/report lưu trong `artifacts/`, `reports/`, `logs/`.

In [ ]:
!python -u scripts/train_rules.py
from pathlib import Path
assert Path('artifacts/mining/fpgrowth_rules.json').is_file(), 'Chưa tạo được file luật FP-Growth.'
!python -u scripts/train_compatibility.py
assert Path('artifacts/compatibility/metadata_baseline/random_forest.joblib').is_file(), 'Huấn luyện compatibility chưa lưu model; không chạy bước đánh giá.'
!python -u scripts/evaluate_recommendation.py
!python -u scripts/run_training_pipeline.py --profile full --start-at embedding

## 5. Kiểm tra kết quả trước khi làm giao diện
Nếu runtime bị ngắt, kiểm tra `reports/pipeline_full.json` và checkpoint trong `artifacts/`; nối lại bằng cell huấn luyện. Chỉ bắt đầu giao diện khi kiểm tra đầu ra hoàn tất và đã xem xét chất lượng/giới hạn metric.

In [ ]:
!python -u scripts/check_training_status.py
from pathlib import Path
for p in [Path('reports/pipeline_full.json'), Path('artifacts/embedding/full/training_report.json')]:
    print('\n###', p, '| exists:', p.exists())
    if p.exists(): print(p.read_text()[:5000])